In [14]:
import logging

In [15]:
from contextlib import contextmanager

In [16]:
import pyodbc

In [17]:
from app.config import DBSettings

In [18]:
logger = logging.getLogger(__name__)

In [19]:
settings = DBSettings.from_env()


In [20]:
def _escape(value: str) -> str:
    """ODBC values containing ; { } = or spaces are wrapped in braces; '}' is escaped as '}}'."""
    return "{" + value.replace("}", "}}") + "}"


In [21]:
def build_connection_string(s: DBSettings) -> str:
    """Build a connection string from the given settings."""
    parts = [
        f"DRIVER={_escape(s.driver)}",
        f"SERVER={_escape(s.server)}",
        f"DATABASE={_escape(s.database)}",
    ]
    if s.trusted_connection:
        parts.append("Trusted_Connection=yes")
    else:
        parts.append(f"UID={_escape(s.user)}")
        parts.append(f"PWD={_escape(s.password)}")
    if s.encrypt:
        parts.append("Encrypt=yes")
    if s.trust_server_certificate:
        parts.append("TrustServerCertificate=yes")
    return ";".join(parts)

In [22]:
@contextmanager
def get_connection():
    """Yield a connection; commit on success, rollback on error, always close."""
    conn = pyodbc.connect(build_connection_string(settings), timeout=5)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

In [24]:
if __name__ == "__main__":
    # Smoke test: run `python -m app.db` to verify DB connectivity
    # BEFORE involving any web framework.
    with get_connection() as conn:
        row = conn.cursor().execute("SELECT DB_NAME(), SUSER_SNAME()").fetchone()
        print(f"Connected to database '{row[0]}' as '{row[1]}'")

Connected to database 'Uni Project' as 'DESKTOP-HQ5M4D6\Shakya'
